In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

from spark_spread import (
    spark_spread,
    dark_spread,
    DEFAULT_COAL_EMISSION_FACTOR
)

In [ ]:
ttf = yf.download("TTF=F", start="2024-01-01")["Close"]
ttf.name = "Gas"
ttf.dropna(inplace=True)
ttf.tail()

In [ ]:
# Coal synthetic relationship
coal_price = ttf * 0.6
coal_price.name = "Coal"

# Gas plant params
heat_rate_gas = 7.5
emission_factor_gas = 0.2

# Coal plant params
heat_rate_coal = 10.0
emission_factor_coal = DEFAULT_COAL_EMISSION_FACTOR  # ≈0.9

# Carbon
carbon_price = 25.0

# Trading signals
long_entry = 30
long_exit = 40

In [ ]:
breakeven = (ttf * heat_rate_gas) + (carbon_price * emission_factor_gas)
power_price = breakeven * 1.10  # synthetic power curve

df = pd.DataFrame({
    "Gas": ttf,
    "Coal": coal_price,
    "Breakeven": breakeven,
    "Power": power_price
})
df.tail()

In [ ]:
df["Spark"] = spark_spread(
    df["Power"], df["Gas"], heat_rate_gas,
    carbon_price=carbon_price,
    emission_factor=emission_factor_gas
)

df["Dark"] = dark_spread(
    df["Power"], df["Coal"], heat_rate_coal,
    carbon_price=carbon_price,
    emission_factor=emission_factor_coal
)

df.tail()

In [ ]:
df["Signal"] = 0
pos = 0

for i in range(len(df)):
    if pos == 0 and df["Spark"].iloc[i] < long_entry:
        pos = 1
    elif pos == 1 and df["Spark"].iloc[i] > long_exit:
        pos = 0
    df.iloc[i, df.columns.get_loc("Signal")] = pos

df.tail()

In [ ]:
df["Daily_PnL"] = df["Signal"].shift(1) * df["Spark"].diff()
df["Cumulative_PnL"] = df["Daily_PnL"].cumsum()

df[["Daily_PnL", "Cumulative_PnL"]].tail()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12,10), sharex=True)

# --- Gas price ---
ax1.plot(df.index, df["Gas"], color="steelblue")
ax1.set_title("TTF Gas Price (€/MWh_th)")
ax1.grid(True, alpha=0.3)

# --- Spark Spread + Signals ---
ax2.plot(df.index, df["Spark"], label="Spark Spread", color="orange")
ax2.axhline(long_entry, color="green", linestyle="--", label="Entry")
ax2.axhline(long_exit, color="red", linestyle="--", label="Exit")
ax2.fill_between(df.index, df["Spark"], long_entry,
                 where=(df["Signal"]==1), alpha=0.2, color="green")
ax2.set_title("Spark Spread + Trading Signals")
ax2.grid(True, alpha=0.3)
ax2.legend()

# --- Cumulative PnL ---
ax3.plot(df.index, df["Cumulative_PnL"], color="purple")
ax3.set_title("Cumulative PnL (Long Spark Spread Strategy)")
ax3.axhline(0, color="black", linestyle="--")
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df.index, df["Spark"], label="Spark (Gas)", color="orange")
plt.plot(df.index, df["Dark"], label="Dark (Coal)", color="brown")
plt.title("Spark vs Dark Spread")
plt.ylabel("€/MWh")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()